In [1]:
import logging

from dotenv import load_dotenv

from utils.data_helpers import initialize_metadata_data, initialize_stock_data

load_dotenv()

logger = logging.getLogger(__name__)


await initialize_stock_data()
await initialize_metadata_data()

2025-11-19 19:51:11,219 - utils.data_helpers - INFO - Initializing stock data mappings...
2025-11-19 19:51:11,227 - utils.api_utils - INFO - Retrieved active equity and sub listing data from cache
2025-11-19 19:51:11,236 - utils.data_helpers - INFO - Stock data initialized: 5451 stocks, 5433 symbols
2025-11-19 19:51:11,236 - utils.data_helpers - INFO - Initializing metadata mappings (20 years of data)...
2025-11-19 19:51:11,247 - utils.api_utils - INFO - Retrieved meta data file for 2005-11-24 to 2025-11-19 from cache
2025-11-19 19:51:11,261 - utils.data_helpers - INFO - Metadata initialized: 22109 items, 4064 unique fincodes


In [2]:
from rag.ingestion.document_fetcher import DocumentFetcher

fetcher = DocumentFetcher()

In [3]:
test_fincode = 103806  # Bajaj Finance Ltd.
docs = await fetcher.get_available_documents(
    fincode=test_fincode,
)
for doc in docs:
    logger.info(f"Document: {doc.filename} ({doc.category}) - {doc.document_date}")

2025-11-19 19:51:13,498 - rag.ingestion.document_fetcher - INFO - Fetching document metadata: fincode=103806, category=None, date_range=2023-11-20 to 2025-11-19
2025-11-19 19:51:13,520 - utils.api_utils - INFO - Retrieved meta data file for 2023-11-20 to 2025-11-19 from cache
2025-11-19 19:51:13,521 - rag.ingestion.document_fetcher - INFO - Found 24 documents matching filters
2025-11-19 19:51:13,522 - __main__ - INFO - Document: 88feb214-165a-4db1-b916-df4fbeae221c.pdf (concall) - 2025-11-04
2025-11-19 19:51:13,522 - __main__ - INFO - Document: af22f3b1-711d-4253-a21f-61afab86ecfe.pdf (concall) - 2025-10-28
2025-11-19 19:51:13,523 - __main__ - INFO - Document: 720e915d-3fc3-40a7-a9ea-0f60e4e0c987.pdf (investor-presentation) - 2025-10-28
2025-11-19 19:51:13,523 - __main__ - INFO - Document: 94216bfc-7da8-47a6-a2a1-1c23381d73e5.pdf (concall) - 2025-09-05
2025-11-19 19:51:13,523 - __main__ - INFO - Document: 1c5b93c8-634c-49fb-a78c-b0668b20f679.pdf (concall) - 2025-08-29
2025-11-19 19:51:

In [4]:
from utils.data_helpers import fincode_to_symbol

company_name = fincode_to_symbol(test_fincode)

In [5]:
company_name

'SRF'

In [6]:
for doc in docs:
    try:
        await fetcher.save_document_to_disk(
            doc.filename, doc.category, f".cache/company_documents/{company_name}"
        )
    except Exception as e:
        logger.error(f"Error saving document {doc.filename} to disk: {e}")
        continue

2025-11-19 19:51:13,988 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=concall&attachmentName=88feb214-165a-4db1-b916-df4fbeae221c.pdf "HTTP/1.1 200 OK"
2025-11-19 19:51:14,064 - rag.ingestion.document_fetcher - INFO - Saved PDF to disk: .cache/company_documents/SRF\88feb214-165a-4db1-b916-df4fbeae221c.pdf
2025-11-19 19:51:14,576 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=concall&attachmentName=af22f3b1-711d-4253-a21f-61afab86ecfe.pdf "HTTP/1.1 200 OK"
2025-11-19 19:51:14,985 - rag.ingestion.document_fetcher - INFO - Saved PDF to disk: .cache/company_documents/SRF\af22f3b1-711d-4253-a21f-61afab86ecfe.pdf
2025-11-19 19:51:15,357 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=investor-presentation&attachmentName=720e915d-3fc3-40a7-a9